# LLM Adaptation Workflow
### Notebook 01b — Neural Networks from Scratch

---

## What this notebook does

Every model in this project — the LLM, the embedding model, the reward model — is a neural network. This notebook builds one from scratch so you understand exactly what is happening inside them.

We'll go in three stages:

1. **A single neuron** — the basic building block
2. **A feedforward neural network** — layers of neurons, trained to classify financial text
3. **A tiny language model** — trained character-by-character on financial text, the same principle as GPT

By the end, when you see `model.generate(...)` in Notebook 04, you'll know exactly what computation is happening under the hood.

We use **raw PyTorch only** — no HuggingFace, no abstractions.

---

## The core idea

A neural network is a function that maps numbers to numbers.

```
input numbers  →  [layers of maths]  →  output numbers
```

The "maths" is just: multiply by some weights, add a bias, apply a non-linearity. Repeat many times. The weights start random. Training adjusts them until the output is useful.

That's it. Everything else — transformers, attention, RLHF — is built on top of this idea.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Set a random seed so results are reproducible
torch.manual_seed(42)

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---

## Part 1 — A single neuron

A neuron takes several numbers as input, multiplies each by a learned weight, sums them up, adds a bias, and applies an activation function.

```
inputs:   x1=0.5,  x2=0.3,  x3=0.9
weights:  w1=0.2,  w2=-0.5, w3=0.8
bias:     b=0.1

output = activation( x1*w1 + x2*w2 + x3*w3 + b )
       = activation( 0.5×0.2 + 0.3×(-0.5) + 0.9×0.8 + 0.1 )
       = activation( 0.1 - 0.15 + 0.72 + 0.1 )
       = activation( 0.77 )
       = ReLU(0.77) = 0.77
```

The weights and bias are the things training will adjust. Everything else is fixed.

In [ ]:
# Build a single neuron manually — no nn.Module, just raw tensors

# Inputs (imagine these are features describing a financial statement)
x = torch.tensor([0.5, 0.3, 0.9])  # e.g. revenue growth, margin, debt ratio (normalised)

# Weights — start random, training will adjust these
w = torch.tensor([0.2, -0.5, 0.8])

# Bias — a constant offset
b = torch.tensor(0.1)

# Linear combination: dot product of inputs and weights, plus bias
z = torch.dot(x, w) + b
print(f"Linear combination z = {z.item():.4f}")

# Activation function: ReLU (Rectified Linear Unit) — the most common
# ReLU simply: output = max(0, z)
# This introduces non-linearity — without it, stacking layers does nothing useful
output = F.relu(z)
print(f"After ReLU:     output = {output.item():.4f}")

print()
print("A negative input:")
z_neg = torch.tensor(-0.5)
print(f"  ReLU({z_neg.item()}) = {F.relu(z_neg).item()}")  # ReLU kills negatives
print()
print("Why activation functions matter:")
print("  Without them, stacking layers = just one big linear transformation.")
print("  With them, we can learn any non-linear pattern.")

In [ ]:
# Visualise the most common activation functions
x_range = torch.linspace(-3, 3, 200)

activations = {
    "ReLU": F.relu(x_range),
    "Sigmoid": torch.sigmoid(x_range),
    "Tanh": torch.tanh(x_range),
    "GELU": F.gelu(x_range),   # used in transformers (e.g. BERT, GPT)
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, (name, vals) in zip(axes, activations.items()):
    ax.plot(x_range.numpy(), vals.numpy(), linewidth=2, color="#4C72B0")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_title(name, fontweight="bold")
    ax.set_xlim(-3, 3)
    ax.grid(True, alpha=0.3)

plt.suptitle("Activation Functions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("ReLU    — simple, fast, the most common in deep networks")
print("Sigmoid — squashes to (0,1), used in binary classification outputs")
print("Tanh    — squashes to (-1,1), used in recurrent networks")
print("GELU    — smooth version of ReLU, used in transformer feed-forward layers")

---

## Part 2 — A feedforward neural network

A neural network is just many neurons organised into layers. The output of one layer becomes the input to the next.

```
Input layer    Hidden layer 1    Hidden layer 2    Output layer
  (3 neurons)    (16 neurons)      (8 neurons)       (1 neuron)

  x1 ─────────────────────────────────────────────── ŷ
  x2 ─── (weights × inputs + bias → activation) ───
  x3 ─────────────────────────────────────────────
```

We'll train this network on a simple financial sentiment task:
**Given text features, predict whether a company's stock direction is positive or negative.**

We'll use simple hand-crafted features (keyword counts) so we can understand the data — in real NLP you'd use embeddings.

In [ ]:
# --- Dataset ---
# Financial headlines with simple bag-of-words features
# Features: counts of positive words, negative words, revenue mentions, beat/miss mentions

# Format: [positive_word_count, negative_word_count, revenue_mentioned, beat_mentioned, miss_mentioned]
#         label: 1 = bullish signal, 0 = bearish signal

raw_data = [
    # (features,                                  label, headline)
    ([3, 0, 1, 1, 0], 1, "Revenue beats estimates, strong quarterly growth"),
    ([0, 3, 1, 0, 1], 0, "Revenue misses estimates, weak demand outlook"),
    ([2, 1, 0, 1, 0], 1, "Earnings beat expectations, raises guidance"),
    ([1, 2, 1, 0, 1], 0, "Sales miss forecasts, lowering annual guidance"),
    ([4, 0, 1, 1, 0], 1, "Record revenue, strong margins, beat on all metrics"),
    ([0, 4, 0, 0, 1], 0, "Weak results, significant miss, cutting costs"),
    ([2, 0, 1, 0, 0], 1, "Revenue growth accelerates, new products launch well"),
    ([0, 2, 0, 0, 1], 0, "Disappointing results, restructuring charges announced"),
    ([3, 1, 1, 1, 0], 1, "Strong earnings beat, raised dividend, revenue up"),
    ([1, 3, 1, 0, 1], 0, "Revenue miss, margin pressure, cuts workforce"),
    ([2, 0, 0, 1, 0], 1, "Beat on EPS, positive outlook, shares rise"),
    ([0, 2, 1, 0, 0], 0, "Revenue declines, market share losses continue"),
    ([3, 0, 0, 1, 0], 1, "Impressive growth, analyst upgrades, strong buy signals"),
    ([1, 2, 0, 0, 1], 0, "Miss on key metrics, regulatory headwinds growing"),
    ([2, 1, 1, 0, 0], 1, "Revenue in line, margins expand, positive signals"),
    ([0, 3, 0, 0, 1], 0, "Significant miss, poor guidance, sell-off continues"),
]

# Convert to tensors
X = torch.tensor([d[0] for d in raw_data], dtype=torch.float32)
y = torch.tensor([d[1] for d in raw_data], dtype=torch.float32)

print(f"Dataset: {len(raw_data)} examples")
print(f"Input shape : {X.shape}  (16 examples × 5 features)")
print(f"Label shape : {y.shape}  (16 labels: 1=bullish, 0=bearish)")
print(f"Class balance: {y.mean():.0%} positive")
print()
print("Features: [positive_words, negative_words, revenue_mentioned, beat_mentioned, miss_mentioned]")

In [ ]:
# --- Build the neural network ---
# nn.Module is PyTorch's base class for all neural networks.
# We define the layers in __init__ and the forward pass in forward().

class FinancialSentimentNet(nn.Module):
    """
    A feedforward network that classifies financial signals as bullish or bearish.
    
    Architecture:
      Input(5) → Linear(16) → ReLU → Linear(8) → ReLU → Linear(1) → Sigmoid
    """
    
    def __init__(self):
        super().__init__()
        
        # nn.Linear(in, out) creates a weight matrix of shape (in × out) and a bias of shape (out,)
        # These are the parameters that training will update
        self.layer1 = nn.Linear(5, 16)   # 5 input features → 16 hidden neurons
        self.layer2 = nn.Linear(16, 8)   # 16 → 8
        self.layer3 = nn.Linear(8, 1)    # 8 → 1 output (probability)
        
    def forward(self, x):
        """
        The forward pass: define how data flows through the network.
        PyTorch automatically handles the backward pass (gradients) for us.
        """
        # Layer 1: linear transformation + ReLU activation
        x = self.layer1(x)   # shape: (batch, 5) → (batch, 16)
        x = F.relu(x)         # apply activation
        
        # Layer 2
        x = self.layer2(x)   # (batch, 16) → (batch, 8)
        x = F.relu(x)
        
        # Output layer: sigmoid squashes output to (0, 1) — a probability
        x = self.layer3(x)   # (batch, 8) → (batch, 1)
        x = torch.sigmoid(x)
        
        return x.squeeze()   # remove the trailing dimension: (batch, 1) → (batch,)


model = FinancialSentimentNet()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Network architecture:")
print(f"  layer1: {5}×{16} weights + {16} biases = {5*16+16} parameters")
print(f"  layer2: {16}×{8} weights + {8} biases = {16*8+8} parameters")
print(f"  layer3: {8}×{1} weights + {1} bias  = {8*1+1} parameters")
print(f"  Total: {total_params} parameters")
print()
print(f"(Compare: TinyLlama has 1,100,000,000 parameters — same idea, much bigger)")

---

## Part 2b — Training: how a network learns

Training a neural network has four steps, repeated thousands of times:

```
1. Forward pass  — run inputs through the network, get predictions
2. Compute loss  — measure how wrong the predictions are
3. Backward pass — compute gradients: how should each weight change to reduce loss?
4. Update weights — move each weight slightly in the direction that reduces loss
```

The **loss** is a number that measures wrongness. We want to minimise it.

The **gradient** of the loss with respect to each weight tells us the direction and magnitude of the steepest increase. We go in the opposite direction — this is **gradient descent**.

PyTorch computes gradients automatically via **autograd** — we just call `loss.backward()`.

In [ ]:
# --- Training loop ---

# Loss function: Binary Cross-Entropy
# For binary classification, BCE measures: how surprised is the model by the true label?
# If the model predicts 0.9 and the true label is 1: low loss (good)
# If the model predicts 0.1 and the true label is 1: high loss (bad)
loss_fn = nn.BCELoss()

# Optimiser: Adam — a smart version of gradient descent
# lr (learning rate): how big a step to take each update. Too high = overshoot, too low = slow.
optimiser = torch.optim.Adam(model.parameters(), lr=0.01)

# Track loss over time so we can visualise learning
loss_history = []

num_epochs = 200  # How many full passes through the training data

print("Training...")
for epoch in range(num_epochs):
    
    # ── Step 1: Forward pass ─────────────────────────────────
    predictions = model(X)           # Run all examples through the network
    loss = loss_fn(predictions, y)   # Compare predictions to true labels
    
    # ── Step 2: Backward pass ────────────────────────────────
    optimiser.zero_grad()  # Clear gradients from previous step (they accumulate otherwise)
    loss.backward()        # Compute gradients: d(loss)/d(each weight)
    
    # ── Step 3: Update weights ───────────────────────────────
    optimiser.step()       # Adjust each weight by -lr × gradient
    
    loss_history.append(loss.item())
    
    if epoch % 40 == 0:
        accuracy = ((predictions > 0.5) == y.bool()).float().mean()
        print(f"  Epoch {epoch:3d}: loss = {loss.item():.4f}  accuracy = {accuracy.item():.0%}")

print()
print(f"Final loss     : {loss_history[-1]:.4f}")
print(f"Final accuracy : {((model(X) > 0.5) == y.bool()).float().mean().item():.0%}")

In [ ]:
# Visualise the learning curve — loss decreasing over time
plt.figure(figsize=(9, 4))
plt.plot(loss_history, color="#4C72B0", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss over Time\n(network is learning — loss decreases as weights improve)", fontweight="bold")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Each step down in the loss curve = the network got a little better at the task.")
print("This same curve is what you see in Notebook 04 when fine-tuning the LLM.")

In [ ]:
# Inspect what the network learned
with torch.no_grad():
    predictions = model(X)

print("Predictions vs true labels:")
print(f"{'Headline':55s} {'True':>6} {'Pred':>7} {'Correct':>8}")
print("-" * 80)
for i, (_, label, headline) in enumerate(raw_data):
    pred = predictions[i].item()
    correct = "✓" if (pred > 0.5) == bool(label) else "✗"
    print(f"{headline[:54]:55s} {label:>6}  {pred:>6.2f}  {correct:>8}")

---

## Part 3 — A tiny language model from scratch

Language models are neural networks trained to **predict the next character (or word) in a sequence**.

That's the whole training objective. At sufficient scale and data, this single task forces a model to learn grammar, facts, reasoning, and style — because you can't predict text well without understanding it.

We'll build a **character-level language model**: given the last N characters of financial text, predict the next character. This is the same principle as GPT — just much, much smaller.

Our training corpus: financial phrases.

In [ ]:
# Training text: financial domain text
# In a real language model, this would be billions of characters.
# We use a small corpus to keep training fast, but the principle is identical.

text = """
revenue grew strongly in the fiscal year driven by product demand.
earnings per share beat analyst estimates by a significant margin.
the company reported record net income for the quarter.
operating margins expanded as cost efficiencies were realised.
revenue declined due to weaker consumer demand and supply constraints.
the board declared a quarterly dividend and announced a share buyback.
net interest income rose as interest rates increased through the year.
cloud services revenue grew strongly driven by enterprise adoption.
the company guided for revenue growth in the range of ten to fifteen percent.
gross margin improved as the product mix shifted toward higher margin services.
capital expenditure increased as the company invested in data centre expansion.
free cash flow generation remained strong supporting continued shareholder returns.
the acquisition was completed and is expected to be accretive to earnings.
currency headwinds reduced reported revenue growth by approximately two percent.
research and development spending increased to drive future product innovation.
""".strip().lower()

print(f"Training corpus: {len(text)} characters")
print(f"Unique characters: {len(set(text))}")
print()
print("Sample:")
print(f"  {text[:120]}...")

In [ ]:
# --- Tokenisation (character level) ---
# Map each unique character to an integer ID, and back.
# In GPT, this is done at the sub-word level (same idea, larger vocabulary).

chars = sorted(set(text))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# Encode the entire corpus as a sequence of integers
encoded = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

print(f"Vocabulary size: {vocab_size} characters")
print(f"Encoded length : {len(encoded)} tokens")
print()
print("First 30 characters and their IDs:")
for ch, idx in zip(text[:30], encoded[:30].tolist()):
    print(f"  '{ch}' → {idx}", end="  ")
print()

In [ ]:
# --- Build training examples ---
# For each position in the text, create an (input, target) pair:
#   input  = the last CONTEXT_LEN characters
#   target = the next character
# 
# Example with CONTEXT_LEN=5:
#   "reven" → 'u'
#   "evenu" → 'e'
#   "venue" → ' '

CONTEXT_LEN = 20  # How many characters of history the model sees at once

inputs  = []
targets = []

for i in range(len(encoded) - CONTEXT_LEN):
    inputs.append(encoded[i : i + CONTEXT_LEN])   # context window
    targets.append(encoded[i + CONTEXT_LEN])       # next character

X_lm = torch.stack(inputs)    # shape: (num_examples, CONTEXT_LEN)
y_lm = torch.stack(targets)   # shape: (num_examples,)

print(f"Training examples: {len(X_lm)}")
print(f"Input shape : {X_lm.shape}  (each row = {CONTEXT_LEN} character IDs)")
print(f"Target shape: {y_lm.shape}  (each entry = the next character ID)")
print()
print("Example:")
example_input = ''.join(idx_to_char[i.item()] for i in X_lm[0])
example_target = idx_to_char[y_lm[0].item()]
print(f"  Input : '{example_input}'")
print(f"  Target: '{example_target}'")

In [ ]:
# --- The language model architecture ---
# 
# Input: a sequence of character IDs
# Step 1: Embedding — convert each character ID to a dense vector
# Step 2: Flatten — treat the whole context window as one long vector
# Step 3: Hidden layers — learn patterns in character sequences
# Step 4: Output — a probability over every possible next character
#
# The key new piece is the Embedding layer:
#   Each character ID maps to a learned vector (just like word embeddings in a real LLM)

EMBED_DIM = 16    # Each character is represented as a 16-dimensional vector
HIDDEN_DIM = 128  # Size of hidden layers

class TinyLanguageModel(nn.Module):
    
    def __init__(self, vocab_size, embed_dim, context_len, hidden_dim):
        super().__init__()
        
        # Embedding: turns each character ID (integer) into a vector
        # Think of it as a lookup table: ID → learned representation
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # After embedding, each context window becomes context_len × embed_dim numbers.
        # We flatten this into a single vector and pass through fully connected layers.
        input_size = context_len * embed_dim
        
        self.fc1 = nn.Linear(input_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, vocab_size)  # output: score for each possible character
        
        self.dropout = nn.Dropout(0.2)  # randomly zero out neurons during training — prevents memorisation
    
    def forward(self, x):
        # x shape: (batch, context_len) — integer character IDs
        
        # Embedding: (batch, context_len) → (batch, context_len, embed_dim)
        x = self.embedding(x)
        
        # Flatten: (batch, context_len, embed_dim) → (batch, context_len × embed_dim)
        x = x.view(x.size(0), -1)
        
        # Hidden layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        
        # Output layer — returns raw scores (logits), not probabilities
        # We don't apply softmax here because the loss function (CrossEntropyLoss) does it internally
        return self.fc3(x)  # shape: (batch, vocab_size)


lm = TinyLanguageModel(vocab_size, EMBED_DIM, CONTEXT_LEN, HIDDEN_DIM).to(device)

lm_params = sum(p.numel() for p in lm.parameters())
print(f"Tiny language model: {lm_params:,} parameters")
print()
print("(TinyLlama is 1.1 billion — same architecture, 10,000x more parameters)")

In [ ]:
# --- Training the language model ---

# CrossEntropyLoss: standard loss for classification over multiple classes
# Here: each next-character prediction is a classification over vocab_size classes
lm_loss_fn = nn.CrossEntropyLoss()
lm_optimiser = torch.optim.Adam(lm.parameters(), lr=3e-3)

# Move data to device
X_lm = X_lm.to(device)
y_lm = y_lm.to(device)

BATCH_SIZE = 64
NUM_EPOCHS = 300
lm_loss_history = []

print(f"Training tiny language model for {NUM_EPOCHS} epochs...")

for epoch in range(NUM_EPOCHS):
    
    # Mini-batch training: instead of using all data at once,
    # sample a random subset (batch) each step — faster and better generalisation
    idx = torch.randperm(len(X_lm))[:BATCH_SIZE]
    X_batch = X_lm[idx]
    y_batch = y_lm[idx]
    
    logits = lm(X_batch)                    # forward pass
    loss = lm_loss_fn(logits, y_batch)      # compute loss
    
    lm_optimiser.zero_grad()
    loss.backward()                          # backward pass — compute gradients
    
    # Gradient clipping: prevent gradients from getting too large and destabilising training
    # This is used in every modern LLM training run
    torch.nn.utils.clip_grad_norm_(lm.parameters(), max_norm=1.0)
    
    lm_optimiser.step()                     # update weights
    
    lm_loss_history.append(loss.item())
    
    if epoch % 60 == 0:
        print(f"  Epoch {epoch:3d}: loss = {loss.item():.4f}")

print(f"\nFinal loss: {lm_loss_history[-1]:.4f}")

In [ ]:
# --- Generate text from the trained model ---
# This is exactly what happens when you call model.generate() on a big LLM.
# The only difference is scale.

def generate_text(model, seed_text, num_chars=200, temperature=0.8):
    """
    Generate text character by character.
    
    temperature: controls randomness
      - Low (0.1):  always picks the most likely next character — repetitive but coherent
      - High (1.5): more random — creative but may produce nonsense
    """
    model.eval()
    
    # Start with the seed text (must be at least CONTEXT_LEN chars)
    seed_text = seed_text.lower()
    if len(seed_text) < CONTEXT_LEN:
        seed_text = seed_text.rjust(CONTEXT_LEN)  # pad with spaces
    
    context = [char_to_idx.get(c, 0) for c in seed_text[-CONTEXT_LEN:]]
    generated = seed_text
    
    with torch.no_grad():
        for _ in range(num_chars):
            x = torch.tensor([context], dtype=torch.long).to(device)
            
            logits = model(x)              # get scores for each possible next character
            logits = logits / temperature  # divide by temperature to control randomness
            probs = F.softmax(logits, dim=-1)  # convert scores → probabilities
            
            # Sample the next character from the probability distribution
            next_id = torch.multinomial(probs, num_samples=1).item()
            next_char = idx_to_char[next_id]
            
            generated += next_char
            context = context[1:] + [next_id]  # slide the context window forward
    
    return generated


# Generate from several seeds
seeds = ["revenue grew", "earnings per share", "the company"]

for seed in seeds:
    print(f"Seed: '{seed}'")
    print("-" * 50)
    generated = generate_text(lm, seed, num_chars=120, temperature=0.7)
    print(generated)
    print()

In [ ]:
# Visualise how temperature affects output
seed = "revenue grew strongly"

print("Effect of temperature on generated text:")
print("(same seed, different randomness)")
print()

for temp in [0.2, 0.7, 1.5]:
    generated = generate_text(lm, seed, num_chars=80, temperature=temp)
    print(f"Temperature = {temp}  (={'deterministic' if temp < 0.5 else 'balanced' if temp < 1.0 else 'random'}):")
    print(f"  {generated}")
    print()

---

## Connecting back to the full workflow

What we just built is the same thing as TinyLlama, GPT-4, and every other LLM — the difference is only scale and architecture refinements.

| What we built | What TinyLlama does |
|---------------|--------------------|
| Character-level vocabulary (~30 chars) | Sub-word vocabulary (~32,000 tokens) |
| 20-character context window | 2,048-token context window |
| Simple feedforward layers | Transformer layers with attention |
| ~60,000 parameters | 1,100,000,000 parameters |
| Trained on 1,500 characters | Trained on 3 trillion tokens |
| Generates plausible financial phrases | Generates fluent, knowledgeable text |

The training loop in Notebook 04 — forward pass, loss, backward, step — is exactly what we did here. The only difference is that the `Trainer` class handles batching and logging for us.

---

## Summary

In this notebook we built:
- A **single neuron** — the basic building block: weights × inputs + bias → activation
- A **feedforward neural network** — layers of neurons trained on financial sentiment classification
- A **tiny language model** — trained to predict the next character in financial text, the same principle as GPT

We ran the full training loop from first principles: forward pass, loss, backward pass, gradient update.

---

▶ **Next: [Notebook 02 — Data Preparation](02_data_preparation.ipynb)**